# REXIA – Responsible and Explainable AI
## Projet 2026 – Partie 1 : Données Tabulaires

**Jeu de données :** `RH_dataset.csv`  
**Objectif :** Prédire si un employé va démissionner dans les 6 prochains mois (label 0/1)

## 0. Imports et chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../RH_dataset.csv', sep=';')
print(f'Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
df.head()

---
## 1. Analyse du jeu de données

### 1.1 Informations générales

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### 1.2 Valeurs manquantes

In [ ]:
missing = df.isnull().sum().rename('nb_manquants')
missing_pct = (df.isnull().mean() * 100).rename('% manquants')
missing_df = pd.concat([missing, missing_pct], axis=1)
print(missing_df[missing_df['nb_manquants'] > 0])
print('\nAucune valeur manquante.' if missing_df['nb_manquants'].sum() == 0 else '')

> **Commentaire :** Si aucune valeur manquante n'est détectée, le jeu de données est complet et ne nécessite pas d'imputation. En cas de valeurs manquantes, des stratégies d'imputation (médiane pour les numériques, mode pour les catégorielles) seraient à envisager.

### 1.3 Exploration des colonnes

In [ ]:
# Colonnes numériques et catégorielles
num_cols = df.select_dtypes(include='number').columns.drop('label', errors='ignore').drop('matricule', errors='ignore').tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Colonnes numériques :', num_cols)
print('Colonnes catégorielles :', cat_cols)

In [ ]:
# Distribution des variables numériques
n_cols = len(num_cols)
fig, axes = plt.subplots(nrows=(n_cols + 2) // 3, ncols=3, figsize=(16, 4 * ((n_cols + 2) // 3)))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], ax=axes[i], kde=True, color='steelblue')
    axes[i].set_title(col)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions des variables numériques', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des variables catégorielles
for col in cat_cols:
    fig, ax = plt.subplots(figsize=(8, 3))
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, palette='muted', ax=ax)
    ax.set_title(f'Distribution – {col}')
    plt.tight_layout()
    plt.show()
    print(df[col].value_counts().to_string(), '\n')

> **Commentaire :** Observer les plages de valeurs, les distributions (asymétriques ou non), la cardinalité des variables catégorielles et la présence éventuelle d'outliers dans les variables numériques.

### 1.4 Distribution de la variable cible (label)

In [ ]:
label_counts = df['label'].value_counts()
label_pct = df['label'].value_counts(normalize=True) * 100

print('Répartition des classes :')
print(pd.DataFrame({'Effectifs': label_counts, '% ': label_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x='label', palette=['steelblue', 'tomato'], ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels(['0 – Resté', '1 – Démissionné'])
ax.set_title('Répartition du label (démission)')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}\n({p.get_height()/len(df)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

> **Commentaire :** Vérifier l'équilibre des classes. Un fort déséquilibre (ex. < 10 % de démissionnaires) impliquerait d'utiliser des techniques de rééquilibrage (SMOTE, class_weight) lors de la phase de modélisation.

### 1.5 Parcours d'un employé démissionnaire vs. non-démissionnaire

In [ ]:
# Sélection d'un exemple par classe
emp_demissionnaire = df[df['label'] == 1].iloc[0]
emp_reste = df[df['label'] == 0].iloc[0]

comparison = pd.DataFrame({
    'Démissionnaire (label=1)': emp_demissionnaire,
    'Non-démissionnaire (label=0)': emp_reste
})
comparison

In [ ]:
# Comparaison des moyennes numériques par label
means_by_label = df.groupby('label')[num_cols].mean().T
means_by_label.columns = ['Non-démissionnaire (0)', 'Démissionnaire (1)']
means_by_label['Différence (1-0)'] = means_by_label['Démissionnaire (1)'] - means_by_label['Non-démissionnaire (0)']
means_by_label.style.background_gradient(subset=['Différence (1-0)'], cmap='RdYlGn_r')

In [ ]:
# Boxplots comparatifs
fig, axes = plt.subplots(nrows=(len(num_cols) + 2) // 3, ncols=3,
                          figsize=(16, 4 * ((len(num_cols) + 2) // 3)))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='label', y=col, palette=['steelblue', 'tomato'], ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xticklabels(['Resté (0)', 'Démissionné (1)'])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des variables numériques par label', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

> **Commentaire :** Analyser les différences entre les deux profils. Par exemple, un employé démissionnaire peut avoir eu une dernière promotion plus ancienne, un salaire inférieur, ou une ancienneté plus faible — autant de signaux potentiellement prédictifs.

### 1.6 Matrice de corrélation

In [ ]:
corr_cols = num_cols + ['label']
corr = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Matrice de corrélation (Pearson)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Corrélations avec le label, triées
corr_with_label = corr['label'].drop('label').sort_values(key=abs, ascending=False)
print('Corrélations avec le label (ordre décroissant) :')
print(corr_with_label.to_string())

> **Commentaire :** La matrice de corrélation permet d'identifier les variables les plus liées au label et de détecter des colinéarités entre prédicteurs. Des corrélations élevées entre variables explicatives peuvent indiquer une redondance d'information (multicolinéarité).

### 1.7 Analyses complémentaires

In [ ]:
# Taux de démission par variable catégorielle
for col in cat_cols:
    taux = df.groupby(col)['label'].mean().sort_values(ascending=False) * 100
    fig, ax = plt.subplots(figsize=(8, 3))
    taux.plot(kind='barh', ax=ax, color='tomato')
    ax.set_xlabel('Taux de démission (%)')
    ax.set_title(f'Taux de démission par « {col} »')
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution de l'âge colorée par label
fig, ax = plt.subplots(figsize=(9, 4))
for label_val, color, name in [(0, 'steelblue', 'Resté'), (1, 'tomato', 'Démissionné')]:
    sns.kdeplot(df[df['label'] == label_val]['Âge (années)'], ax=ax,
                label=name, color=color, fill=True, alpha=0.4)
ax.set_title('Distribution de l\'âge par label')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Salaire vs Ancienneté, coloré par label
fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(
    df['Ancienneté groupe (années)'], df['Salaire (Euros)'],
    c=df['label'], cmap='coolwarm', alpha=0.3, s=10
)
plt.colorbar(scatter, ax=ax, label='Label (0=resté, 1=démissionné)')
ax.set_xlabel('Ancienneté groupe (années)')
ax.set_ylabel('Salaire (Euros)')
ax.set_title('Salaire vs Ancienneté, coloré par label')
plt.tight_layout()
plt.show()

> **Commentaire :** Ces visualisations permettent d'identifier des segments à risque de démission (ex. certains établissements, familles d'emploi, tranches d'âge). Elles serviront de base à la conception du modèle prédictif.

---
## 2. Analyse des variables sensibles et biais potentiels

### 2.1 Identification des variables sensibles et proxies

Dans le contexte des RH et de l'IA responsable, une **variable sensible** est une variable dont l'utilisation dans un modèle prédictif pourrait conduire à une discrimination illégale ou éthiquement inacceptable (au sens du RGPD et de la législation anti-discrimination).

Un **proxy** est une variable corrélée à une caractéristique sensible, permettant d'inférer indirectement cette dernière.

| Variable | Type | Justification |
|---|---|---|
| **Âge (années)** | **Sensible directe** | L'âge est explicitement protégé par le droit du travail (discrimination par l'âge). |
| **Statut marital** | **Sensible directe** | Peut être corrélé au sexe (congé maternité/paternité) et à la situation familiale. |
| **Parent** | **Sensible / Proxy** | Variable binaire indiquant si l'employé est parent. Peut constituer un proxy pour le sexe (si les femmes sont sur-représentées parmi les parents qui démissionnent). |
| **Salaire (Euros)** | **Proxy potentiel** | Peut refléter des inégalités de genre ou d'origine. Corrélé à l'âge et à l'ancienneté. |
| **Etablissement** | **Proxy potentiel** | Peut être corrélé à la nationalité, l'origine géographique ou le niveau socio-économique. |
| **Famille d'emploi** | **Proxy potentiel** | Certains secteurs sont fortement genrés (ex. infirmier/infirmière) et pourraient révéler indirectement le sexe. |
| **Niveau hiérarchique** | **Proxy potentiel** | Peut refléter des discriminations systémiques liées au genre ou à l'origine. |
| **matricule** | **Identifiant** | Identifiant unique : doit être exclu des modèles pour éviter le surapprentissage et la réidentification. |

In [ ]:
# Analyse de la corrélation entre variables sensibles et le label
sensitive_vars = ['Âge (années)', 'Parent', 'Salaire (Euros)', 'Niveau hiérarchique']

fig, axes = plt.subplots(1, len(sensitive_vars), figsize=(16, 4))
for i, col in enumerate(sensitive_vars):
    sns.boxplot(data=df, x='label', y=col, palette=['steelblue', 'tomato'], ax=axes[i])
    axes[i].set_title(f'{col}\nvs label')
    axes[i].set_xticklabels(['Resté', 'Démissionné'])
plt.suptitle('Variables sensibles vs label', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Taux de démission par statut marital et statut parental
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

taux_marital = df.groupby('Statut marital')['label'].mean() * 100
taux_marital.sort_values().plot(kind='barh', ax=axes[0], color='mediumpurple')
axes[0].set_xlabel('Taux de démission (%)')
axes[0].set_title('Taux de démission par statut marital')

taux_parent = df.groupby('Parent')['label'].mean() * 100
taux_parent.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'])
axes[1].set_xlabel('Parent (0=Non, 1=Oui)')
axes[1].set_ylabel('Taux de démission (%)')
axes[1].set_title('Taux de démission selon le statut parental')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

> **Commentaire :** Si le taux de démission diffère significativement selon le statut marital ou parental, ces variables pourraient constituer des proxies de genre. Leur inclusion dans un modèle prédictif devra être questionnée au regard des principes de non-discrimination et de fairness algorithmique (ex. test de parité démographique, égalité des chances).

In [ ]:
# Corrélation entre variables sensibles (pour détecter les proxies)
sensitive_and_num = [c for c in ['Âge (années)', 'Salaire (Euros)', 'Ancienneté groupe (années)',
                                   'Niveau hiérarchique', 'Parent', 'Début de contrat (années)'] if c in df.columns]

fig, ax = plt.subplots(figsize=(8, 6))
corr_sens = df[sensitive_and_num].corr()
sns.heatmap(corr_sens, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Corrélations entre variables potentiellement sensibles')
plt.tight_layout()
plt.show()

### 2.2 Synthèse et recommandations

**Variables à traiter avec précaution :**
- **`Âge (années)`** : directement sensible → envisager de la supprimer ou de mesurer l'impact sur l'équité du modèle.
- **`Statut marital`** et **`Parent`** : proxies potentiels → analyser les disparités de prédiction entre sous-groupes.
- **`Famille d'emploi`** : proxy de genre possible → surveiller les métriques de fairness par modalité.
- **`matricule`** : identifiant → **exclure obligatoirement** du modèle.

**Recommandations IA responsable :**
1. Exclure `matricule` des features.
2. Mesurer les métriques de fairness (parité démographique, égalité des chances) pour les variables sensibles.
3. Evaluer l'impact de la suppression des variables sensibles sur les performances du modèle (trade-off performance / équité).
4. Appliquer des techniques d'atténuation des biais si nécessaire (re-weighting, adversarial debiasing).

---
## 3. Apprentissage automatique *(à compléter lors de la prochaine séance)*

Cette section sera complétée lors de la prochaine séance de TP.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

# Préparation des données
df_model = df.drop(columns=['matricule'])

X = df_model.drop(columns=['label'])
y = df_model['label']

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(include='number').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train : {X_train.shape}, Test : {X_test.shape}')

In [ ]:
# Arbre de décision (interprétable)
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'))
])
dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)

print('=== Arbre de décision ===')
print(classification_report(y_test, y_pred_dt, target_names=['Resté (0)', 'Démissionné (1)']))
print(f'AUC-ROC : {roc_auc_score(y_test, dt_pipeline.predict_proba(X_test)[:, 1]):.4f}')

In [ ]:
# Random Forest
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10,
                                           random_state=42, class_weight='balanced', n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Resté (0)', 'Démissionné (1)']))
print(f'AUC-ROC : {roc_auc_score(y_test, rf_pipeline.predict_proba(X_test)[:, 1]):.4f}')

In [ ]:
# Matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_dt, y_pred_rf],
    ['Arbre de décision', 'Random Forest']
):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Resté (0)', 'Démissionné (1)']).plot(ax=ax, colorbar=False)
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# Importance des variables (Random Forest)
rf_model = rf_pipeline.named_steps['classifier']
ohe_features = rf_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_features).tolist()
all_features = num_features + ohe_features

importances = pd.Series(rf_model.feature_importances_, index=all_features)
top_20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
top_20.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 variables importantes – Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

> **Commentaire :** Les variables les plus importants selon le Random Forest peuvent être comparées aux analyses exploratoires de la Partie 1. Si des variables sensibles apparaissent en tête, cela justifie une analyse d'équité approfondie (XAI / SHAP).